<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/50_build_app_artefacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Build artefacts from extracted runs (keywords models) ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, shutil, math

PROJECT_DIR = Path("/content/drive/MyDrive/gt-markets")
SRC_EXTRACT = PROJECT_DIR/"app-demo"/"extracted"    # daily/weekly folders
DST_ARTE    = PROJECT_DIR/"AppDemo"/"artefacts"     # merge with baseline outputs
DATA_PROCESSED = PROJECT_DIR/"data"/"processed"

ASSETS=["GOLD","BTC","OIL","USDCNY"]
FREQ_DIRS={"daily":"D","weekly":"W"}
ANNUALIZATION={"D":252,"W":52}; COST_BPS=5

def to_equity(close,pos,freq):
    ret=close.pct_change().fillna(0); pos=pos.fillna(0).clip(0,1)
    strat=(pos.shift(1)*ret)-pos.diff().abs().fillna(0)*(COST_BPS/1e4)
    eq=(1+strat).cumprod(); ann=ANNUALIZATION[freq]
    mu=strat.mean()*ann; sd=strat.std()*math.sqrt(ann) if strat.std()>0 else np.nan
    sharpe=mu/sd if sd and sd>0 else np.nan; maxdd=(eq/eq.cummax()-1).min()
    return eq,{"Return_Ann":mu,"Vol_Ann":sd,"Sharpe":sharpe,"MaxDD":maxdd}

def infer_asset(name):
    name=name.lower()
    if "gold" in name or "gc=f" in name: return "GOLD"
    if "btc" in name: return "BTC"
    if "oil" in name or "cl=f" in name: return "OIL"
    if "usdcny" in name: return "USDCNY"
    return None

for sub, F in FREQ_DIRS.items():
    src=SRC_EXTRACT/sub
    if not src.exists(): continue
    for csv in src.rglob("*.csv"):
        asset=infer_asset(csv.name) or infer_asset(str(csv))
        if not asset: continue
        try: df=pd.read_csv(csv)
        except: continue
        if "Date" not in df.columns: continue
        df["Date"]=pd.to_datetime(df["Date"],errors="coerce"); df=df.dropna(subset=["Date"])
        # Determine position
        pos=None
        if "position" in df.columns: pos=df["position"].astype(float)
        elif "prob_up" in df.columns: pos=(df["prob_up"]>0.5).astype(float)
        else: continue
        if "Close" not in df.columns: continue
        close=pd.to_numeric(df["Close"],errors="coerce"); close.index=df["Date"]
        # Strategy name from file
        strat="KW_"+csv.stem.split("_")[0].upper()
        out_dir=DST_ARTE/asset; (out_dir/"figs").mkdir(parents=True,exist_ok=True)
        # Write signals
        sig=pos.diff().fillna(0).replace({1.0:1,-1.0:0}).astype(int)
        pd.DataFrame({"Date":df["Date"],"signal":sig,"position":pos.astype(int),
                      "Close":close.values,"strategy":strat}).to_csv(
            out_dir/f"signals_{strat}_{F}.csv",index=False)
        # Metrics + fig
        eq, mets=to_equity(close,pos,F)
        ax=eq.plot(figsize=(6,3),title=f"{asset}-{strat}-{F}"); ax.grid(True,alpha=0.3)
        ax.get_figure().savefig(out_dir/"figs"/f"{strat}_{F}.png",dpi=150,bbox_inches="tight")
        plt.close(ax.get_figure())
        row={"asset":asset,"freq":F,"strategy":strat,**mets}
        out_csv=out_dir/f"metrics_keywords_{F}.csv"
        if out_csv.exists(): old=pd.read_csv(out_csv); pd.concat([old,pd.DataFrame([row])]).to_csv(out_csv,index=False)
        else: pd.DataFrame([row]).to_csv(out_csv,index=False)

print("✅ Keyword artefacts merged into", DST_ARTE)


Mounted at /content/drive
✅ Keyword artefacts merged into /content/drive/MyDrive/gt-markets/AppDemo/artefacts
